# 02 — Transformacja danych

## Cel

W tym notebooku przygotuję oczyszczone dane do dalszej analizy i modelu.

Tutaj zajmiemy się:
- wyborem niewielkiej liczby sensownych cech,
- brakami pozostawionymi po czyszczeniu,
- kodowaniem kategorii,
- prostą transformacją danych.

Nie będą potrzebne wszystkie 55 kolumn.


## 1. Import bibliotek i wczytanie danych

In [1]:
# Path służy do tworzenia i obsługi ścieżek do plików i folderów.
from pathlib import Path

import pandas as pd


# Ścieżka do danych, które zostały zapisane po etapie czyszczenia.
DATA_PATH = Path("data/02_po_czyszczeniu.csv")

# Ścieżka do pliku z danymi przygotowanymi bezpośrednio do modelu.
OUTPUT_MODEL_PATH = Path("data/03_do_modelu.csv")

# Ścieżka do pełnego datasetu po transformacji.
# Ten plik wykorzystamy później m.in. podczas EDA.
OUTPUT_FULL_PATH = Path("data/03_pelny_po_transformacji.csv")

# Nazwa kolumny, którą docelowo chcemy przewidywać.
TARGET = "interest_rate"


# Wczytuję dane zapisane po zakończeniu czyszczenia.
df = pd.read_csv(DATA_PATH)

# Sprawdzam rozmiar datasetu po wczytaniu.
print("Kształt:", df.shape)

# Wyświetlam pierwsze 5 wierszy, żeby upewnić się,
# że dane zostały poprawnie wczytane.
df.head()


Kształt: (10000, 55)


,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,debt_to_income_joint,...,sub_grade,issue_month,loan_status,initial_listing_status,disbursement_method,balance,paid_total,paid_principal,paid_interest,paid_late_fees
0,global config engineer,3.0,NJ,MORTGAGE,90000.0,Verified,18.01,NaN,NaN,NaN,...,C3,Mar-2018,Current,whole,Cash,27015.86,1999.33,984.14,1015.19,0.0
1,warehouse office clerk,10.0,HI,RENT,40000.0,Not Verified,5.04,NaN,NaN,NaN,...,C1,Feb-2018,Current,whole,Cash,4651.37,499.12,348.63,150.49,0.0
2,assembly,3.0,WI,RENT,40000.0,Source Verified,21.15,NaN,NaN,NaN,...,D1,Feb-2018,Current,fractional,Cash,1824.63,281.80,175.37,106.43,0.0
3,customer service,1.0,PA,RENT,30000.0,Not Verified,10.16,NaN,NaN,NaN,...,A3,Jan-2018,Current,whole,Cash,18853.26,3312.89,2746.74,566.15,0.0
4,security supervisor,10.0,CA,RENT,35000.0,Verified,57.96,57000.0,Verified,37.66,...,C3,Mar-2018,Current,whole,Cash,21430.15,2324.65,1569.85,754.80,0.0


## 2. Wybór cech

W tym miejscu wybierzemy kilka prostych cech, które:
- są zrozumiałe,
- były dostępne przy udzielaniu pożyczki,
- nie są informacją powstałą później podczas spłaty !!!

Ten wybór dopracujemy wspólnie podczas pracy nad tym notebookiem.


In [2]:
# Wybieram niewielki zestaw cech, które mogą mieć związek
# z oprocentowaniem pożyczki i są łatwe do interpretacji.

wybrane_kolumny = [
    "loan_amount",       # kwota pożyczki
    "term",              # okres pożyczki w miesiącach
    "annual_income",     # roczny dochód
    "debt_to_income",    # stosunek zadłużenia do dochodu
    "emp_length",        # staż pracy
    "verified_income",   # sposób weryfikacji dochodu
    "homeownership",     # sytuacja mieszkaniowa
    TARGET,              # interest_rate
]

# Tworzę osobny DataFrame zawierający tylko kolumny,
# które wykorzystam podczas przygotowania danych do modelu.
df_model = df[wybrane_kolumny].copy()

df_model.head()


,loan_amount,term,annual_income,debt_to_income,emp_length,verified_income,homeownership,interest_rate
0,28000,60,90000.0,18.01,3.0,Verified,MORTGAGE,14.07
1,5000,36,40000.0,5.04,10.0,Not Verified,RENT,12.61
2,2000,36,40000.0,21.15,3.0,Source Verified,RENT,17.09
3,21600,36,30000.0,10.16,1.0,Not Verified,RENT,6.72
4,23000,36,35000.0,57.96,10.0,Verified,RENT,14.07


### Wybrałem kilka łatwych do interpretacji cech dostępnych przy udzielaniu pożyczki.
Nie wykorzystuję kolumn opisujących późniejszą spłatę pożyczki, ponieważ informacje
te nie były dostępne w momencie ustalania oprocentowania.

## 3. Obsługa brakujących wartości

### 3.1. Sprawdzenie brakujących wartości

In [3]:
# Sprawdzam liczbę brakujących wartości
# tylko w kolumnach wybranych do modelu.
braki_model = df_model.isna().sum()

# Obliczam również procent braków w każdej kolumnie.
procent_brakow_model = (
    braki_model / len(df_model) * 100
).round(2)

tabela_brakow_model = pd.DataFrame({
    "braki": braki_model,
    "procent_brakow": procent_brakow_model
})

tabela_brakow_model[
    tabela_brakow_model["braki"] > 0
]


,braki,procent_brakow
debt_to_income,24,0.24
emp_length,817,8.17


### 3.2. Obliczenie median

In [4]:
# Obliczam medianę dla kolumn numerycznych,
# w których występują brakujące wartości.
mediana_dti = df_model["debt_to_income"].median()
mediana_emp_length = df_model["emp_length"].median()

print("Mediana debt_to_income:", mediana_dti)
print("Mediana emp_length:", mediana_emp_length)

Mediana debt_to_income: 17.57
Mediana emp_length: 6.0


### 3.3. Uzupełnienie braków medianą

In [5]:
# Uzupełniam brakujące wartości medianą.
# Mediana jest mniej wrażliwa na wartości odstające niż średnia.

df_model["debt_to_income"] = (
    df_model["debt_to_income"]
    .fillna(mediana_dti)
)

df_model["emp_length"] = (
    df_model["emp_length"]
    .fillna(mediana_emp_length)
)

### 3.4. Kontrola po uzupełnieniu braków

In [6]:
# Sprawdzam, czy po imputacji zostały jeszcze braki.
df_model.isna().sum()

loan_amount        0
term               0
annual_income      0
debt_to_income     0
emp_length         0
verified_income    0
homeownership      0
interest_rate      0
dtype: int64

## 4. Kodowanie zmiennych kategorycznych

### 4.1. Sprawdzenie kategorii

In [7]:
# Sprawdzam, jakie wartości występują
# w wybranych kolumnach tekstowych.

print("verified_income:")
print(df_model["verified_income"].value_counts())

print("\nhomeownership:")
print(df_model["homeownership"].value_counts())


verified_income:
verified_income
Source Verified    4116
Not Verified       3594
Verified           2290
Name: count, dtype: int64

homeownership:
homeownership
MORTGAGE    4789
RENT        3858
OWN         1353
Name: count, dtype: int64


### 4.2. One-hot encoding

In [8]:
# Zamieniam zmienne tekstowe na kolumny numeryczne 0/1.
# pd.get_dummies() tworzy osobną kolumnę dla każdej kategorii.
#
# drop_first=True pomija pierwszą kategorię z każdej zmiennej,
# ponieważ jedna kategoria może pełnić rolę kategorii odniesienia.
#
# dtype=int powoduje zapis nowych kolumn jako 0 i 1.

df_model = pd.get_dummies(
    df_model,
    columns=["verified_income", "homeownership"],
    drop_first=True,
    dtype=int
)

df_model.head()

,loan_amount,term,annual_income,debt_to_income,emp_length,interest_rate,verified_income_Source Verified,verified_income_Verified,homeownership_OWN,homeownership_RENT
0,28000,60,90000.0,18.01,3.0,14.07,0,1,0,0
1,5000,36,40000.0,5.04,10.0,12.61,0,0,0,1
2,2000,36,40000.0,21.15,3.0,17.09,1,0,0,1
3,21600,36,30000.0,10.16,1.0,6.72,0,0,0,1
4,23000,36,35000.0,57.96,10.0,14.07,0,1,0,1


### 4.3. Kontrola po kodowaniu

In [9]:
# Sprawdzam typy danych po kodowaniu.
# W danych przygotowywanych do modelu nie powinny już
# występować kolumny tekstowe.

df_model.dtypes

loan_amount                          int64
term                                 int64
annual_income                      float64
debt_to_income                     float64
emp_length                         float64
interest_rate                      float64
verified_income_Source Verified      int64
verified_income_Verified             int64
homeownership_OWN                    int64
homeownership_RENT                   int64
dtype: object

### Wnioski po kodowaniu

Kolumny `verified_income` i `homeownership` były zmiennymi tekstowymi,
dlatego model regresji liniowej nie mógłby wykorzystać ich bezpośrednio.

Za pomocą `pd.get_dummies()` zostały zamienione na kolumny zawierające wartości
0 i 1. Po kodowaniu dane wybrane do modelu mają postać numeryczną.

## 5. Skalowanie / standaryzacja

Standaryzacja polega na przekształceniu cech numerycznych tak, aby miały
porównywalną skalę.

W tym projekcie używam zwykłej regresji liniowej (`LinearRegression`),
która nie wymaga standaryzacji danych do poprawnego działania.

Dlatego na tym etapie nie skaluję cech. Pozostawiam wartości w ich
oryginalnych jednostkach, co ułatwi również późniejszą interpretację
współczynników modelu.

## 6. Zapis danych po transformacji

Po zakończeniu transformacji zapisuję dwa pliki.

`03_do_modelu.csv` zawiera tylko cechy wybrane do modelu.
Braki zostały uzupełnione, a zmienne tekstowe zakodowane do postaci numerycznej.

`03_pelny_po_transformacji.csv` zachowuje pełny dataset w bardziej czytelnej
postaci. Wykorzystam go później podczas eksploracyjnej analizy danych (EDA).

In [10]:
# Tworzę kopię pełnego datasetu.
# W tej wersji zachowuję oryginalne kolumny tekstowe,
# dzięki czemu dane będą łatwiejsze do analizy podczas EDA.
df_pelny = df.copy()

# Uzupełniam w pełnym datasecie te same braki,
# które wcześniej uzupełniłem w danych przygotowywanych do modelu.
df_pelny["debt_to_income"] = (
    df_pelny["debt_to_income"]
    .fillna(mediana_dti)
)

df_pelny["emp_length"] = (
    df_pelny["emp_length"]
    .fillna(mediana_emp_length)
)


In [11]:
# Zapisuję dane przygotowane bezpośrednio do modelu.
df_model.to_csv(
    OUTPUT_MODEL_PATH,
    index=False
)

# Zapisuję pełny dataset po podstawowej transformacji.
# Ten plik wykorzystam później w notebooku EDA.
df_pelny.to_csv(
    OUTPUT_FULL_PATH,
    index=False
)

print("Zapisano:", OUTPUT_MODEL_PATH)
print("Zapisano:", OUTPUT_FULL_PATH)

Zapisano: data/03_do_modelu.csv
Zapisano: data/03_pelny_po_transformacji.csv


In [12]:
# Kontrola
print("Dane do modelu:", df_model.shape)
print("Pełne dane do EDA:", df_pelny.shape)

Dane do modelu: (10000, 10)
Pełne dane do EDA: (10000, 55)


### Podsumowanie transformacji

Do modelu wybrałem niewielki zestaw łatwych do interpretacji cech.

Brakujące wartości w `debt_to_income` oraz `emp_length` uzupełniłem medianą.
Zmienne kategoryczne `verified_income` i `homeownership` zakodowałem metodą
one-hot encoding.

Nie zastosowałem standaryzacji, ponieważ w tym projekcie będę używał
zwykłej regresji liniowej i chcę zachować cechy w ich oryginalnych jednostkach.

Przygotowane dane zapisałem do osobnych plików dla EDA i dla modelu.